# yt-clips Colab Worker
## Runtime → Change runtime type → T4 GPU

Run all cells in order. Worker will start and show tunnel URL.

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1: Mount Google Drive
# ═══════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Find project folder
import os
from pathlib import Path

project_path = None
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        project_path = p
        break

if project_path:
    os.chdir(project_path)
    print(f"Working directory: {project_path}")
else:
    os.chdir("/content")
    print("No yt-clips folder in Drive. Upload files manually via sidebar.")
    !ls -la

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2: Install ALL Dependencies
# ═══════════════════════════════════════════════
import subprocess
import sys

def run_cmd(cmd, desc):
    print(f"→ {desc}...")
    subprocess.run(cmd, shell=True, capture_output=True)

# System deps
run_cmd("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg nasm yasm build-essential > /dev/null 2>&1", 
        "Installing ffmpeg, aria2, build tools")

# Deno (bot bypass)
run_cmd("curl -fsSL https://deno.land/x/install/install.sh | sh > /dev/null 2>&1", 
        "Installing Deno")
os.environ["PATH"] += ":/root/.deno/bin"

# Python base packages
run_cmd("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy filterpy scipy "
        "google-api-python-client google-auth-httplib2 google-auth-oauthlib requests Pillow "
        "google-genai google-generativeai openai python-dotenv pytest pytest-timeout 2>&1 | tail -1",
        "Installing Python packages")

# GPU packages - PROPERLY with CUDA index
run_cmd("pip install -q ultralytics torch --extra-index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1",
        "Installing PyTorch + YOLOv8 (CUDA)")
run_cmd("pip install -q gfpgan basicsr > /dev/null 2>&1", 
        "Installing GFPGAN")

# Localtunnel
run_cmd("npm install -g localtunnel > /dev/null 2>&1", 
        "Installing localtunnel")

# Verify GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null", 
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu if gpu else 'NONE - use T4 GPU runtime!'}")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3: Write Colab Config
# ═══════════════════════════════════════════════
import yaml
from pathlib import Path

# Create folders
for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs"]:
    Path(folder).mkdir(exist_ok=True)

config = """
paths:
  input:       input/
  temp:        temp/
  transcripts: transcripts/
  highlights:  highlights/
  shorts:      shorts/
  logs:        logs/
download:
  format: "bv*[height<=1080]+ba/b[height<=1080]/bv*+ba/b"
  concurrent_fragments: 32
  output_filename: "video.mp4"
  use_aria2c: true
transcription:
  model: "base"
  language: "hi"
  device: "cuda"
  compute_type: "float16"
highlight:
  use_ai_refinement: true
  audio_energy_threshold: 0.3
  min_duration: 10
  max_duration: 29
  merge_gap: 8
  max_clips: 10
premium:
  enabled: true
  face_enhancement: true
  frame_interpolation: true
layout:
  has_facecam: true
  facecam: {x: 0, y: 540, width: 320, height: 180}
  facecam_output_height: 400
  gameplay_output_height: 1520
export:
  width: 1080
  height: 1920
  fps: 60
  crf: 18
  encoder: "h264_nvenc"
  enable_variable_speed: true
youtube:
  privacy_status: "private"
  upload_enabled: false
ai:
  provider: "gemini"
  api_key: ""
  model: "gemini-2.0-flash-lite"
thumbnail:
  enabled: true
testing:
  enabled: false
logging:
  level: "INFO"
  log_file: "logs/pipeline.log"
"""

with open("config.yaml", "w") as f:
    f.write(config.strip())
print("config.yaml written (GPU mode, premium enabled)")

# Load API key from Colab secrets
try:
    from google.colab import userdata
    key = userdata.get("AI_API_KEY")
    if key:
        os.environ["AI_API_KEY"] = key
        with open("config.yaml") as f:
            cfg = yaml.safe_load(f)
        cfg["ai"]["api_key"] = key
        with open("config.yaml", "w") as f:
            yaml.dump(cfg, f, default_flow_style=False)
        print("AI_API_KEY loaded from secrets")
except:
    print("No AI_API_KEY in secrets - add via 🔑 icon or ignore")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4: Start Worker + Tunnel
# ═══════════════════════════════════════════════
import time

# Kill old processes
get_ipython().system_raw("pkill -f 'python watcher.py' 2>/dev/null || true")
get_ipython().system_raw("pkill -f 'lt --port' 2>/dev/null || true")
time.sleep(1)

# Start watcher
print("Starting watcher.py...")
get_ipython().system_raw("python watcher.py > watcher.log 2>&1 &")
time.sleep(2)

# Start tunnel
print("Starting localtunnel...")
get_ipython().system_raw("lt --port 5000 > tunnel.log 2>&1 &")
time.sleep(5)

# Show tunnel URL
with open("tunnel.log") as f:
    for line in f:
        if "://" in line.strip():
            tunnel_url = line.strip()
            with open("colab_url.txt", "w") as out:
                out.write(tunnel_url)
            print(f"\n🔗 TUNNEL URL: {tunnel_url}")
            print(f"   (saved to colab_url.txt)")
            break

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5: Monitor (keep running)
# ═══════════════════════════════════════════════
print("=" * 55)
print("  WORKER IS ONLINE!")
print("=" * 55)
print("\nOn your Mac, run:")
print('  ./automate.sh "https://youtu.be/VIDEO_ID"')
print("  → Select option 2 (Remote Run)")
print("\nMonitoring watcher.log...")

import time
try:
    while True:
        time.sleep(20)
        if Path("watcher.log").exists():
            with open("watcher.log") as f:
                lines = f.readlines()
                for l in lines[-5:]:
                    if l.strip():
                        print(l.strip())
except KeyboardInterrupt:
    print("\nStopping...")